
# 4CBOn2 Kaggle4b — Full-Source Candidate Builder

**This is a builder notebook, not the competition submission notebook.** It packages a complete Franzen/Duck source tree for you to inspect and test. It never contacts Kaggle, runs the candidate, scores it, submits it, or pushes it to GitHub.

## The two modes

### B-1 — Offline packer (no model call)
Use this after you have a patch you already trust or want to inspect. Load the Franzen source, upload or paste the `.diff` from Kaggle4, and optionally add logs and a short reason. Kaggle4b checks that the patch applies, performs syntax checks on changed Python/JSON files, then makes a ZIP of the **whole source tree with the patch applied** plus a change manifest. Logs are hashed for provenance only; they are not sent anywhere in B-1.

### B-2 — Live code-maker (Gemini)
Use this when you want Gemini to propose the edits from pasted logs and a design directive. The default inside Colab is `google.colab.ai` (OAuth; no key). If you enter a Gemini API key, Kaggle4b uses that key instead. It sends the selected source excerpts, design directive, and pasted logs to Gemini. It then applies only exact edits to selected existing files, checks syntax and public top-level API compatibility, and packages the whole source tree. It does **not** execute the code.

## Which mode when?

- **B-1** when Kaggle4 has already produced a patch and you want a full-tree ZIP without another model call. This is the safer, more reproducible packaging step.
- **B-2** when you want this notebook to ask Gemini to create the patch. Use it only when you are comfortable sending the selected code excerpts and logs to Gemini.
- In both modes, inspect the manifest and test the ZIP in a separate Kaggle notebook/copy. **You** decide whether to submit.

## Source lineage and safety

- The older v1/v3 runs used `mohamtur1/arc-agi-3-agent`; the 25.83 run you described used the Franzen/Duck Milestone 2 notebook. This builder targets the Franzen/Duck source family, pinned by default to `b2fee614056f0609b94e214f03372f15894c00fe`. Upload the exact source ZIP if your Kaggle copy differs from that pin.
- The baseline is loaded from the pinned GitHub commit only if you do not upload a ZIP. There is **no Kaggle API/fetch** and no access to your Kaggle account or run logs. Paste only the logs you want B-2 to use; do not paste secrets.
- The optional API key is used in memory for B-2 only. It is not written to files or the manifest. Leave it blank in Colab to use Colab OAuth.
- Syntax/API checks are static checks only. Candidate source is never imported or executed.

## Manual hand-off

`4CBOn2_Kaggle4b.ipynb` is the packer. Its ZIP is a source tree, **not itself a Kaggle submission notebook**. Download the ZIP, put/test those source files in your Franzen Milestone 2 Kaggle notebook, compare the offline practice result with your baseline, and submit manually only if you choose.


In [ ]:

%pip -q install gradio requests


In [ ]:

# Core setup. This cell does not call Gemini and does not connect to Kaggle.
import ast
import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import tempfile
import time
import uuid
import zipfile
from pathlib import Path, PurePosixPath

import requests
import gradio as gr

try:
    import google.colab.ai as _colab_ai
except Exception:
    _colab_ai = None

FRANZEN_REPO_URL = "https://github.com/tonghuikang/daniel-franzen-arc-agi-3.git"
FRANZEN_PIN = "b2fee614056f0609b94e214f03372f15894c00fe"
DEFAULT_TARGETS = """ARC3-Inference/inference/agent/python_tool_sandbox.py:150-190
ARC3-Inference/inference/agent/prompts.py:130-155
ARC3-Inference/inference/agent/tool_agent.py:297-308
ARC3-Inference/inference/agent/priority_scheduler.py:70-134
ARC3-Inference/inference/framework/solver.py:276-301
ARC3-Inference/README.md:76-103"""
ALLOWED_SUFFIXES = {".py", ".md", ".json", ".toml", ".yaml", ".yml", ".txt", ".sh"}
MAX_SOURCE_UNPACKED_BYTES = 250_000_000
MAX_SOURCE_ZIP_ENTRIES = 20_000
MAX_CONTEXT_CHARS = 70_000
MAX_TARGET_FILES = 8
MAX_PATCH_BYTES = 2_000_000
WORK_ROOT = Path.cwd() / "kaggle4b_work"
OUTPUT_ROOT = Path.cwd() / "kaggle4b_candidates"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Kaggle4b ready. No Kaggle connection, candidate execution, submission, or score call is configured.")
print("Colab OAuth available:", _colab_ai is not None)


In [ ]:

def _git(command, timeout=300):
    result = subprocess.run(command, capture_output=True, text=True, timeout=timeout)
    if result.returncode:
        detail = (result.stderr or result.stdout or "command failed")[-700:]
        raise RuntimeError(detail)
    return result.stdout.strip()


def _upload_path(upload):
    if upload is None:
        return ""
    if isinstance(upload, dict):
        upload = upload.get("path") or upload.get("name") or ""
    elif hasattr(upload, "name"):
        upload = upload.name
    return os.fspath(upload) if isinstance(upload, (str, os.PathLike)) else ""


def _safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > MAX_SOURCE_ZIP_ENTRIES:
            raise ValueError("Source ZIP exceeds the 20,000-entry limit.")
        if sum(info.file_size for info in entries) > MAX_SOURCE_UNPACKED_BYTES:
            raise ValueError("Source ZIP expands beyond the 250 MB safety limit.")
        for info in entries:
            name = info.filename.replace("\\", "/")
            rel = PurePosixPath(name)
            if rel.is_absolute() or any(part == ".." for part in rel.parts):
                raise ValueError("Source ZIP contains an unsafe path.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("Source ZIP contains a symbolic link; extraction refused.")
            parts = [part for part in rel.parts if part not in ("", ".")]
            target = destination.joinpath(*parts)
            try:
                target.resolve().relative_to(destination)
            except ValueError:
                raise ValueError("Source ZIP path escapes the extraction directory.")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
            perms = (info.external_attr >> 16) & 0o777
            if perms:
                target.chmod(perms)


def _find_repo_root(search_root):
    search_root = Path(search_root).resolve()
    for marker in search_root.rglob("tool_agent.py"):
        if marker.is_symlink():
            continue
        if (marker.parent.name == "agent" and marker.parent.parent.name == "inference"
                and marker.parent.parent.parent.name == "ARC3-Inference"):
            root = marker.parents[3]
            if (root / "ARC3-Inference" / "inference" / "agent" / "prompts.py").is_file():
                return root
    raise ValueError("ZIP/source does not contain the expected Franzen ARC3-Inference tree.")


def _copy_clean_tree(source, destination):
    ignored = shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".pytest_cache", ".mypy_cache",
                                     ".ruff_cache", ".venv", "node_modules", ".DS_Store")
    shutil.copytree(source, destination, ignore=ignored, symlinks=True)
    for path in Path(destination).rglob("*"):
        if path.is_symlink():
            raise ValueError("Baseline contains a symbolic link; refusing to package it safely.")


def _tree_hash(root):
    root = Path(root).resolve()
    digest = hashlib.sha256()
    ignored_dirs = {".git", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache", ".venv", "node_modules"}
    for path in sorted(root.rglob("*")):
        if any(part in ignored_dirs for part in path.parts):
            continue
        if path.is_symlink():
            raise ValueError("Baseline contains a symbolic link; refusing to hash/package it.")
        if not path.is_file():
            continue
        rel = path.relative_to(root).as_posix()
        digest.update(rel.encode("utf-8")); digest.update(b"\0")
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
    return digest.hexdigest()


def load_franzen_source(source_zip=None, commit=FRANZEN_PIN):
    """Load uploaded Franzen source ZIP, or the pinned GitHub commit; never queries Kaggle."""
    upload = _upload_path(source_zip)
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    if upload:
        archive_path = Path(upload)
        if not archive_path.is_file() or archive_path.suffix.lower() != ".zip":
            raise ValueError("Choose a .zip source archive.")
        archive_sha = hashlib.sha256(archive_path.read_bytes()).hexdigest()
        destination = WORK_ROOT / ("baseline_upload_" + archive_sha[:16])
        marker = destination / "ARC3-Inference" / "inference" / "agent" / "tool_agent.py"
        if not marker.is_file():
            if destination.exists():
                shutil.rmtree(destination)
            stage = Path(tempfile.mkdtemp(prefix="source_zip_", dir=WORK_ROOT))
            try:
                unpacked = stage / "unpacked"
                _safe_extract_zip(archive_path, unpacked)
                found = _find_repo_root(unpacked)
                _copy_clean_tree(found, destination)
            finally:
                shutil.rmtree(stage, ignore_errors=True)
        root = _find_repo_root(destination)
        state = {"root": str(root), "origin": "uploaded_source_zip", "commit": "",
                 "archive_sha256": archive_sha, "tree_sha256": _tree_hash(root)}
        label = "uploaded ZIP sha256 " + archive_sha[:12]
    else:
        commit = str(commit or "").strip().lower()
        if not re.fullmatch(r"[0-9a-f]{40}", commit):
            raise ValueError("Enter a full 40-character commit SHA or upload a source ZIP.")
        destination = WORK_ROOT / ("baseline_commit_" + commit[:16])
        marker = destination / "ARC3-Inference" / "inference" / "agent" / "tool_agent.py"
        head = ""
        if marker.is_file() and (destination / ".git").is_dir():
            try:
                head = _git(["git", "-C", str(destination), "rev-parse", "HEAD"], timeout=30).lower()
            except Exception:
                head = ""
        if head != commit:
            if destination.exists():
                shutil.rmtree(destination)
            destination.mkdir(parents=True, exist_ok=True)
            _git(["git", "init", "--quiet", str(destination)])
            _git(["git", "-C", str(destination), "remote", "add", "origin", FRANZEN_REPO_URL])
            _git(["git", "-C", str(destination), "fetch", "--depth=1", "origin", commit])
            _git(["git", "-C", str(destination), "checkout", "--quiet", "--detach", "FETCH_HEAD"])
            head = _git(["git", "-C", str(destination), "rev-parse", "HEAD"], timeout=30).lower()
            if head != commit:
                raise RuntimeError("Fetched GitHub commit did not match the requested pin.")
        root = _find_repo_root(destination)
        state = {"root": str(root), "origin": FRANZEN_REPO_URL, "commit": commit,
                 "archive_sha256": "", "tree_sha256": _tree_hash(root)}
        label = "commit " + commit
    files = sorted(p.relative_to(root).as_posix() for p in (root / "ARC3-Inference").rglob("*.py"))
    preview = "\n".join(files[:50])
    if len(files) > 50:
        preview += f"\n… and {len(files)-50} more Python files"
    return state, f"Loaded Franzen baseline: {label}; tree sha256 {state['tree_sha256'][:12]}.", preview


In [ ]:

def _target_context(repo_root, target_specs):
    specs = [line.strip() for line in str(target_specs or "").splitlines() if line.strip()]
    if not specs:
        specs = [line.strip() for line in DEFAULT_TARGETS.splitlines() if line.strip()]
    if not 1 <= len(specs) <= MAX_TARGET_FILES:
        raise ValueError(f"Choose 1–{MAX_TARGET_FILES} target paths/ranges.")
    root = Path(repo_root).resolve()
    selected, sections, total = {}, [], 0
    for spec in specs:
        match = re.fullmatch(r"(.*?)(?::([0-9]+)-([0-9]+))?", spec)
        rel_text = (match.group(1) if match else spec).strip()
        start = int(match.group(2)) if match and match.group(2) else None
        end = int(match.group(3)) if match and match.group(3) else None
        rel = PurePosixPath(rel_text.replace("\\", "/"))
        if rel.is_absolute() or any(p in ("", ".", "..") for p in rel.parts):
            raise ValueError(f"Unsafe target path: {rel_text}")
        if rel.suffix.lower() not in ALLOWED_SUFFIXES:
            raise ValueError(f"Unsupported target file type: {rel_text}")
        path = (root / Path(*rel.parts)).resolve()
        try:
            path.relative_to(root)
        except ValueError:
            raise ValueError(f"Target escapes source root: {rel_text}")
        if not path.is_file() or path.is_symlink():
            raise ValueError(f"Target file missing or unsafe: {rel_text}")
        key = rel.as_posix()
        if key in selected:
            raise ValueError(f"Duplicate target path: {key}")
        text = path.read_text(encoding="utf-8")
        lines = text.splitlines(keepends=True)
        if start is None:
            if len(text) > 50_000:
                raise ValueError(f"{key} is large; give a line range such as {key}:20-80.")
            start, end, excerpt = 1, max(1, len(lines)), text
        else:
            if end is None or start < 1 or end < start or end > len(lines):
                raise ValueError(f"Invalid line range for {key}; file has {len(lines)} lines.")
            excerpt = "".join(lines[start-1:end])
        if not excerpt:
            raise ValueError(f"Selected range is empty: {key}")
        total += len(excerpt)
        if total > MAX_CONTEXT_CHARS:
            raise ValueError("Selected source context exceeds 70,000 characters; reduce the target ranges.")
        selected[key] = {"path": path, "original": text, "excerpt": excerpt,
                         "start_line": start, "end_line": end}
        sections.append(f"=== {key}, editable context lines {start}-{end} ===\n{excerpt}")
    return selected, "\n\n".join(sections), specs


def _safe_rel_path(path_text):
    rel = PurePosixPath(str(path_text).replace("\\", "/"))
    if rel.is_absolute() or not rel.parts or any(part in ("", ".", "..") for part in rel.parts):
        raise ValueError(f"Unsafe patch path: {path_text}")
    return rel.as_posix()


def _parse_patch_paths(diff_text):
    paths = set()
    for line in diff_text.splitlines():
        if line.startswith("diff --git "):
            match = re.fullmatch(r"diff --git a/(.+) b/(.+)", line)
            if not match:
                raise ValueError("Patch has an unsupported or quoted diff header; use a standard unified diff.")
            paths.add(_safe_rel_path(match.group(1)))
            paths.add(_safe_rel_path(match.group(2)))
        elif line.startswith("--- ") or line.startswith("+++ "):
            raw = line[4:].split("\t", 1)[0].strip()
            if raw == "/dev/null":
                continue
            if raw.startswith("a/") or raw.startswith("b/"):
                raw = raw[2:]
            elif raw.startswith('"'):
                raise ValueError("Quoted patch paths are not supported on this mobile-friendly path.")
            paths.add(_safe_rel_path(raw))
    if not paths:
        raise ValueError("No standard unified-diff file paths were found.")
    return paths


def _snapshot_tree(root):
    root = Path(root).resolve()
    snapshot = {}
    ignored = {".git", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache", ".venv", "node_modules"}
    for path in sorted(root.rglob("*")):
        if any(part in ignored for part in path.parts):
            continue
        if path.is_symlink():
            raise ValueError("Candidate tree contains a symbolic link; packaging refused.")
        if path.is_file():
            rel = path.relative_to(root).as_posix()
            snapshot[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
    return snapshot


def _changed_paths(before, after):
    return sorted(path for path in set(before) | set(after) if before.get(path) != after.get(path))


def _syntax_check(root, changed_paths):
    checks = []
    for rel in changed_paths:
        path = Path(root) / Path(*PurePosixPath(rel).parts)
        if not path.exists():
            continue
        if path.suffix.lower() == ".py":
            text = path.read_text(encoding="utf-8")
            ast.parse(text, filename=rel)
            compile(text, rel, "exec")  # compile only; never executes the candidate
        elif path.suffix.lower() == ".json":
            json.loads(path.read_text(encoding="utf-8"))
    checks.append("Changed Python files: AST/compile syntax only; changed JSON files: parsed")
    return checks


def _public_top_level_api(path):
    tree = ast.parse(Path(path).read_text(encoding="utf-8"), filename=str(path))
    api = {}
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and not node.name.startswith("_"):
            args = node.args
            signature = (tuple(a.arg for a in args.posonlyargs), tuple(a.arg for a in args.args),
                         args.vararg.arg if args.vararg else None,
                         tuple(a.arg for a in args.kwonlyargs), args.kwarg.arg if args.kwarg else None)
            api[node.name] = ("function", signature)
        elif isinstance(node, ast.ClassDef) and not node.name.startswith("_"):
            api[node.name] = ("class", ())
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                if isinstance(target, ast.Name) and not target.id.startswith("_"):
                    api[target.id] = ("value", ())
    return api


def _top_level_api_check(baseline_root, candidate_root, changed_paths):
    critical = ["ARC3-Inference/inference/agent/tool_agent.py",
                "ARC3-Inference/inference/framework/solver.py"]
    for rel in critical:
        if not (Path(candidate_root) / rel).is_file():
            raise ValueError(f"Top-level API check failed: required module missing: {rel}")
    modules = sorted(set(p for p in changed_paths if p.endswith(".py")) | set(critical))
    preserved = 0
    added = []
    removed = []
    for rel in modules:
        old_path, new_path = Path(baseline_root) / rel, Path(candidate_root) / rel
        if not old_path.is_file() or not new_path.is_file():
            if old_path.is_file() != new_path.is_file():
                removed.append(f"{rel}: module added/removed")
            continue
        old_api, new_api = _public_top_level_api(old_path), _public_top_level_api(new_path)
        for name, signature in old_api.items():
            if name not in new_api:
                removed.append(f"{rel}: {name}")
            elif new_api[name] != signature:
                removed.append(f"{rel}: {name} signature/type changed")
            else:
                preserved += 1
        added.extend(f"{rel}: {name}" for name in new_api if name not in old_api)
    if removed:
        raise ValueError("Top-level API check failed; existing public names/signatures changed: " + "; ".join(removed[:20]))
    return {"result": "passed", "modules_checked": modules,
            "public_symbols_preserved": preserved, "new_public_symbols": added}


In [ ]:

def _sha_text(text):
    return hashlib.sha256(str(text or "").encode("utf-8")).hexdigest()


def _make_run_id(baseline_state, signature):
    salt = f"{baseline_state.get('tree_sha256','')}|{signature}|{time.time_ns()}"
    return time.strftime("%Y%m%d_%H%M%S", time.gmtime()) + "_" + hashlib.sha256(salt.encode()).hexdigest()[:8]


def _zip_full_tree(root, zip_path):
    root = Path(root).resolve()
    ignored = {".git", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache", ".venv", "node_modules"}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(root.rglob("*")):
            if any(part in ignored for part in path.parts):
                continue
            if path.is_symlink():
                raise ValueError("Refusing to put a symbolic link in the source ZIP.")
            if not path.is_file():
                continue
            rel = path.relative_to(root).as_posix()
            info = zipfile.ZipInfo(rel)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = (stat.S_IFREG | (path.stat().st_mode & 0o777)) << 16
            with path.open("rb") as stream:
                archive.writestr(info, stream.read())


def _emit_artifacts(candidate_root, baseline_root, baseline_state, mode, changed_paths,
                    reasons, logs, design="", patch_sha256="", model_route="",
                    checks=None, api_report=None):
    run_id = _make_run_id(baseline_state, mode + patch_sha256 + design)
    out_dir = OUTPUT_ROOT / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir / "candidate_full_franzen_source.zip"
    manifest_path = out_dir / "candidate_change_manifest.json"
    before, after = _snapshot_tree(baseline_root), _snapshot_tree(candidate_root)
    changed_paths = _changed_paths(before, after)
    files = []
    for rel in changed_paths:
        files.append({"path": rel, "why": reasons.get(rel, "Changed by the supplied patch/design."),
                      "baseline_sha256": before.get(rel, ""), "candidate_sha256": after.get(rel, "")})
    manifest = {
        "product": "4CBOn2 Kaggle4b full-source candidate",
        "run_id": run_id,
        "mode": mode,
        "status": "candidate_only_not_submitted",
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "baseline": {"origin": baseline_state.get("origin", ""),
                     "commit": baseline_state.get("commit", ""),
                     "source_archive_sha256": baseline_state.get("archive_sha256", ""),
                     "tree_sha256": baseline_state.get("tree_sha256", "")},
        "model_route": model_route or "none_offline",
        "changed_files": files,
        "checks": checks or [],
        "top_level_api_check": api_report or {"result": "not_run_in_B1"},
        "patch_sha256": patch_sha256,
        "logs_sha256": _sha_text(logs),
        "design_sha256": _sha_text(design),
        "not_done": ["candidate code not executed", "game tests not run", "no score measured",
                     "no Kaggle API/fetch", "nothing submitted", "nothing pushed to GitHub"],
        "artifacts": [zip_path.name, manifest_path.name],
    }
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    _zip_full_tree(candidate_root, zip_path)
    return str(zip_path), str(manifest_path), manifest


def _baseline_ready(state):
    if not isinstance(state, dict) or not state.get("root"):
        raise ValueError("Load the pinned Franzen source or upload its ZIP first.")
    root = Path(state["root"])
    if not root.is_dir() or not (root / "ARC3-Inference/inference/agent/tool_agent.py").is_file():
        raise ValueError("Loaded Franzen source is unavailable; reload the source ZIP/pin.")
    return root


In [ ]:
def build_b1_offline_packer(source_state, logs, patch_file, pasted_patch, rationale):
    """Apply an already-reviewed unified diff; no model call is made in this function."""
    stage_root = None
    try:
        baseline_root = _baseline_ready(source_state)
        uploaded_patch = _upload_path(patch_file)
        if uploaded_patch:
            patch_text = Path(uploaded_patch).read_text(encoding="utf-8")
        else:
            patch_text = str(pasted_patch or "")
        if not patch_text.strip():
            raise ValueError("Upload a .diff/.patch file or paste its text.")
        if len(patch_text.encode("utf-8")) > MAX_PATCH_BYTES:
            raise ValueError("Patch exceeds the 2 MB safety limit.")
        allowed_paths = _parse_patch_paths(patch_text)
        run_id = _make_run_id(source_state, _sha_text(patch_text))
        stage_root = WORK_ROOT / ("b1_stage_" + run_id)
        if stage_root.exists():
            shutil.rmtree(stage_root)
        _copy_clean_tree(baseline_root, stage_root)
        patch_path = WORK_ROOT / ("b1_patch_" + run_id + ".diff")
        patch_path.write_text(patch_text, encoding="utf-8")
        try:
            _git(["git", "init", "--quiet", str(stage_root)])
            _git(["git", "-C", str(stage_root), "apply", "--check", str(patch_path)])
            _git(["git", "-C", str(stage_root), "apply", str(patch_path)])
        finally:
            patch_path.unlink(missing_ok=True)
        before, after = _snapshot_tree(baseline_root), _snapshot_tree(stage_root)
        changed = _changed_paths(before, after)
        if not changed:
            raise ValueError("Patch applied but produced no source changes.")
        unexpected = sorted(set(changed) - allowed_paths)
        if unexpected:
            raise ValueError("Patch changed files outside its declared diff paths: " + ", ".join(unexpected))
        checks = ["Unified diff passed git apply --check", "Patch applied to a staging copy"]
        checks += _syntax_check(stage_root, changed)
        why = str(rationale or "Applied the user-supplied patch; no rationale was entered.").strip()
        reasons = {rel: why for rel in changed}
        zip_path, manifest_path, _ = _emit_artifacts(
            stage_root, baseline_root, source_state, "B1_offline_packer", changed, reasons,
            logs=logs, patch_sha256=_sha_text(patch_text), checks=checks,
        )
        return (f"B-1 complete. Full source ZIP and manifest are ready. No model call was made; "
                f"{len(changed)} file(s) changed. Syntax checks passed; candidate not executed or submitted.",
                zip_path, manifest_path)
    except Exception as exc:
        return f"B-1 stopped safely: {exc}", None, None
    finally:
        if stage_root is not None:
            shutil.rmtree(stage_root, ignore_errors=True)


In [ ]:
def _extract_json_object(text):
    text = str(text or "").strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
                if isinstance(value, dict):
                    return value
            except json.JSONDecodeError:
                continue
    raise ValueError("Gemini response did not contain a valid JSON object.")


def call_gemini(prompt, api_key=""):
    """Use a pasted Gemini key if present; otherwise use Colab google.colab.ai OAuth."""
    key = str(api_key or "").strip()
    if key:
        model = "gemini-2.5-flash"
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
        payload = {"contents": [{"parts": [{"text": prompt}]}],
                   "generationConfig": {"temperature": 0.1, "maxOutputTokens": 8192,
                                        "responseMimeType": "application/json"}}
        try:
            response = requests.post(url, json=payload, timeout=300,
                                     headers={"Content-Type": "application/json", "x-goog-api-key": key})
        except Exception as exc:
            # Never include the request URL or key in an error/status message.
            raise RuntimeError(f"Gemini API request failed ({type(exc).__name__}); key value was not logged.")
        if response.status_code != 200:
            raise RuntimeError(f"Gemini API returned HTTP {response.status_code}; response body/key omitted.")
        data = response.json()
        candidates = data.get("candidates") or [{}]
        parts = (candidates[0].get("content") or {}).get("parts") or []
        text = "".join(str(part.get("text", "")) for part in parts)
        if not text.strip():
            raise RuntimeError("Gemini returned an empty response.")
        return text, "Gemini REST API key", model

    ai = _colab_ai
    if ai is None:
        try:
            import google.colab.ai as ai
        except Exception:
            ai = None
    if ai is None:
        raise RuntimeError("No key was entered and google.colab.ai is unavailable. Open in Colab or enter a Gemini key.")
    try:
        models = ai.list_models()
        model = next((str(item) for item in models if "gemini" in str(item).lower()), None)
        if not model:
            model = str(models[0]) if models else "google/gemini-2.5-flash"
        chunks = []
        for chunk in ai.generate_text(prompt=prompt, model_name=model, stream=True):
            if chunk is not None:
                chunks.append(str(chunk))
        text = "".join(chunks)
        if not text.strip():
            raise RuntimeError("google.colab.ai returned an empty response.")
        return text, "google.colab.ai OAuth", model
    except Exception as exc:
        raise RuntimeError(f"Colab Gemini call failed ({type(exc).__name__}); try again or enter a key.")


def _apply_exact_edits(baseline_root, selected, edits):
    if not isinstance(edits, list) or not 1 <= len(edits) <= 12:
        raise ValueError("Gemini must return 1–12 exact edits; no candidate was packaged.")
    candidate_text = {rel: row["original"] for rel, row in selected.items()}
    reasons = {}
    for index, edit in enumerate(edits, 1):
        if not isinstance(edit, dict):
            raise ValueError(f"Edit {index} is not a JSON object.")
        rel = _safe_rel_path(edit.get("path", ""))
        old, new = edit.get("old"), edit.get("new")
        why = str(edit.get("why") or "Reason not supplied.").strip()
        if rel not in selected:
            raise ValueError(f"Edit {index} targets an unselected file: {rel}")
        if not isinstance(old, str) or not old or not isinstance(new, str):
            raise ValueError(f"Edit {index} requires non-empty old text and string new text.")
        if old not in selected[rel]["excerpt"]:
            raise ValueError(f"Edit {index} is outside the selected context range for {rel}.")
        count = candidate_text[rel].count(old)
        if count != 1:
            raise ValueError(f"Edit {index} matches {count} times in {rel}; exact unique match required.")
        candidate_text[rel] = candidate_text[rel].replace(old, new, 1)
        if why not in reasons.setdefault(rel, []):
            reasons[rel].append(why)
    stage = Path(tempfile.mkdtemp(prefix="b2_candidate_", dir=WORK_ROOT))
    stage.rmdir()  # copytree creates its destination directory
    _copy_clean_tree(baseline_root, stage)
    changed = []
    for rel, text in candidate_text.items():
        original = selected[rel]["original"]
        if text != original:
            target = stage / Path(*PurePosixPath(rel).parts)
            target.write_text(text, encoding="utf-8")
            changed.append(rel)
    if not changed:
        shutil.rmtree(stage, ignore_errors=True)
        raise ValueError("Gemini edits produced no changes.")
    reasons = {rel: " ".join(reasons.get(rel, [])) for rel in changed}
    return stage, sorted(changed), reasons


def build_b2_live_code_maker(source_state, logs, design, target_specs, api_key):
    try:
        baseline_root = _baseline_ready(source_state)
        design = str(design or "").strip()
        if not design:
            raise ValueError("Enter a design directive first.")
        selected, source_context, specs = _target_context(baseline_root, target_specs)
        logs = str(logs or "")[:12_000]
        task = f"""You are a careful multi-file code editor for the Franzen/Duck ARC3-Inference source.
Make the smallest safe source change implementing the design. Preserve existing public APIs and signatures.
Do not change model weights, model identity, scoring/submission behavior, network access, or competition protocol.
The tool will create a candidate ZIP only. Your code will NOT be executed, tested, scored, or submitted here.

Return ONLY valid JSON with this schema:
{{"edits":[{{"path":"exact repo-relative path","old":"exact unique text from the excerpt","new":"replacement text","why":"short reason this file changes"}}]}}
Use 1–12 edits. Only edit the listed existing files and supplied line ranges. Do not add/delete files or return a full file. Each old value must appear exactly once in that file and inside the displayed excerpt. If the design cannot be implemented safely, return {{"edits":[]}}.

DESIGN DIRECTIVE:
{design[:16000]}

PASTED RUN LOGS (evidence, not permission to alter score/submission paths):
{logs}

BASELINE: {source_state.get('origin','')} @ {source_state.get('commit') or source_state.get('archive_sha256','')}
TREE SHA256: {source_state.get('tree_sha256','')}
TARGETS: {json.dumps(specs, ensure_ascii=False)}
SOURCE EXCERPTS:
{source_context}
"""
        raw, model_route, model_name = call_gemini(task, api_key)
        parsed = _extract_json_object(raw)
        edits = parsed.get("edits") if isinstance(parsed, dict) else None
        stage_root, changed, reasons = _apply_exact_edits(baseline_root, selected, edits)
        try:
            checks = _syntax_check(stage_root, changed)
            api_report = _top_level_api_check(baseline_root, stage_root, changed)
            checks.append("Public top-level names/signatures preserved in edited modules and core entrypoints")
            checks.append("Candidate source was not imported or executed")
            zip_path, manifest_path, _ = _emit_artifacts(
                stage_root, baseline_root, source_state, "B2_live_code_maker", changed, reasons,
                logs=logs, design=design, model_route=f"{model_route} ({model_name})",
                checks=checks, api_report=api_report,
            )
        finally:
            shutil.rmtree(stage_root, ignore_errors=True)
        return (f"B-2 complete via {model_route}. {len(changed)} file(s) changed; syntax and static API checks passed. "
                "Candidate not executed, scored, or submitted.", zip_path, manifest_path)
    except Exception as exc:
        return f"B-2 stopped safely: {exc}", None, None


In [ ]:

def _load_source_ui(source_zip, commit):
    try:
        state, status, preview = load_franzen_source(source_zip, commit)
        return state, status, preview
    except Exception as exc:
        return {}, f"Source not loaded: {exc}", ""

with gr.Blocks(title="4CBOn2 Kaggle4b — Full-Source Candidate Builder", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 4CBOn2 Kaggle4b\nLoad the Franzen baseline, then choose **B-1 Offline packer** or **B-2 Live code-maker**. Outputs are candidate files only.")
    source_state = gr.State({})
    with gr.Group():
        gr.Markdown("### 1 · Load the source baseline")
        with gr.Row():
            source_zip = gr.File(label="Optional exact Franzen source ZIP (recommended if your Kaggle copy differs)",
                                 file_types=[".zip"], type="filepath")
            source_commit = gr.Textbox(label="Pinned source commit", value=FRANZEN_PIN, lines=1)
        load_source_btn = gr.Button("Load Franzen source", variant="secondary")
        source_status = gr.Textbox(label="Baseline identity", lines=2, interactive=False)
        source_preview = gr.Textbox(label="Source file preview", lines=5, interactive=False)

    with gr.Tabs():
        with gr.Tab("B-1 · Offline packer"):
            gr.Markdown("No Gemini call. Upload/paste an already-reviewed Kaggle4 unified diff. The logs are recorded by hash only and are not sent anywhere in this mode.")
            b1_logs = gr.Textbox(label="Optional run logs (B-1 only records a hash)", lines=5)
            b1_patch_file = gr.File(label="Kaggle4 patch file (.diff/.patch)",
                                    file_types=[".diff", ".patch", ".txt"], type="filepath")
            b1_patch_text = gr.Textbox(label="Or paste the unified diff here (mobile-friendly)", lines=10)
            b1_reason = gr.Textbox(label="Why this patch was chosen (added to manifest)", lines=2)
            b1_button = gr.Button("Apply patch and package full source tree", variant="primary")
            b1_status = gr.Textbox(label="B-1 status", lines=3, interactive=False)
            b1_zip = gr.File(label="Full Franzen source tree with patch applied (.zip)")
            b1_manifest = gr.File(label="Change manifest (.json)")

        with gr.Tab("B-2 · Live code-maker"):
            gr.Markdown("Gemini receives the design directive, pasted logs, and selected source excerpts. A filled API-key box takes priority; if blank, Colab uses `google.colab.ai` OAuth. No candidate code is run.")
            b2_logs = gr.Textbox(label="Run logs / evidence to send to Gemini", lines=6)
            b2_design = gr.Textbox(label="Design directive", lines=5, placeholder="Describe the exact change you want the agent source to make.")
            b2_targets = gr.Textbox(label="Existing files/ranges allowed for edits (one per line)",
                                    value=DEFAULT_TARGETS, lines=7)
            b2_key = gr.Textbox(label="Optional Gemini API key (leave blank for Colab OAuth)",
                                type="password", lines=1, placeholder="Key is used in memory only; never put it in logs.")
            b2_button = gr.Button("Ask Gemini to edit and package full source tree", variant="primary")
            b2_status = gr.Textbox(label="B-2 status", lines=3, interactive=False)
            b2_zip = gr.File(label="Full Franzen source tree with proposed edits (.zip)")
            b2_manifest = gr.File(label="Change manifest (.json)")

    load_source_btn.click(fn=_load_source_ui, inputs=[source_zip, source_commit],
                          outputs=[source_state, source_status, source_preview])
    b1_button.click(fn=build_b1_offline_packer,
                    inputs=[source_state, b1_logs, b1_patch_file, b1_patch_text, b1_reason],
                    outputs=[b1_status, b1_zip, b1_manifest])
    b2_button.click(fn=build_b2_live_code_maker,
                    inputs=[source_state, b2_logs, b2_design, b2_targets, b2_key],
                    outputs=[b2_status, b2_zip, b2_manifest])

demo.queue(default_concurrency_limit=1).launch(inline=True, share=False, show_error=True)
